# Download GOES (WFABBA) Heat Detection Data from WIFIRE Firemap
### What this notebook does:

1. Search for GOES heat detections by datetime range
2. Download all detection features from the WIFIRE Firemap Geoserver
3. Visualize heat detections on a Folium map
4. Save as JSON, CSV, and/or zipped Shapefile


### Data source:

* WIFIRE Firemap - GOES heat detection data
* Layer: ssec_wfabba_goes_no_low_code
* Contains actual mapped detections with timestamps

### Requirements:

* Internet connection
* Fire name
* Fire start/stop datetimes

# Dependencies

In [ ]:
import requests
import json
import sys
from datetime import datetime, timezone
import folium
import os

# Fire Data Collection


Testing the Geoserver API Query

In [ ]:
# ========== USER INPUTS ==========

fire_name = "Border 2"  # Your chosen fire name will be used for download's filename

# Bounding Box
	# Syntax: "LowerLeft_Longitude,LowerLeft_Latitude,UpperRight_Longitude,UpperRight_Latitude,'EPSG:Code'"
	# For a complete CA BBOX, enter: "-124.5,32.45,-114.0,42.10,'EPSG:4326'"
LL_lon = -116.93191
LL_lat = 32.56482
UR_lon = -116.81826
UR_lat = 32.67451
bbox = f"{LL_lon},{LL_lat},{UR_lon},{UR_lat},'EPSG:4326'"
map_center = [(LL_lat+UR_lat)/2.0, (LL_lon+UR_lon)/2.0]

# Specify a datetime (yyyy, m, d, h, m, s)
dt_start = datetime(2025, 1, 22, 0, 0, 1)    # Note: the first available GOES detection starts from (2017, 5, 20, 21, 35, 0)
dt_end = datetime(2025, 2, 1, 23, 59, 59)   # Note: Large datetime ranges (ex: more than 1 year) may overtax your computer's resources

# ==================================

In [ ]:
# Convert datetime to UTC
dt_start_utc = dt_start.replace(tzinfo=timezone.utc).isoformat(sep='T').replace('+00:00', 'z')
dt_end_utc = dt_end.replace(tzinfo=timezone.utc).isoformat(sep='T').replace('+00:00', 'z')

# GeoServer WFS URL configuration
url = "https://sdge.sdsc.edu/geoserver/wfs"
params = {
    'service': 'WFS',
    'version': '1.0.0',
    'request': 'GetFeature',
    'maxFeatures': '100000',
    'exceptions': 'application/json',
    'typeName': 'WIFIRE:ssec_wfabba_goes_no_low_code',
    'CQL_FILTER': f"data_time BETWEEN '{dt_start_utc}' AND '{dt_end_utc}' AND BBOX(geom,{bbox})",
    'outputFormat': 'application/json'
}

# Request the data
r = requests.get(url, params=params, timeout=60)
if r.status_code != 200:
    print(r.text)
    sys.exit(1)

j = r.json()
f = j.get("features", [])

if not f:
    raise ValueError(
        f"No GOES dections found for '{fire_name}'.\n"
        f"Check that the datetime range is valid.\n"
        f"Hint: the first available GOES detection starts from 2016-08-11T09:20:00Z"
    )

print(f"✓ Found {len(f)} detections \n")

# View the JSON
print(json.dumps(j,indent=2))   # WARNING: Large datetime ranges may overtax your computer's resources

In [ ]:
# # For full Traceback 
# %tb

Previewing the JSON response in GIS

In [ ]:
# Create map object
m = folium.Map(location=map_center, tiles='cartodbpositron', zoom_start=11)

# Add 1000m Radius Circle Marker for GOES GeoJson features
features = folium.features.GeoJson(j,
    marker=folium.Circle(
        radius=1000, # Radius in meters
        fill=True,
        fill_color="blue",
        fill_opacity=0.2),
        tooltip = folium.GeoJsonTooltip(
            fields=['id', 'satellite', 'data_time', 'frp'])
).add_to(m)

# Preview map
m   # WARNING: Large datetime ranges may overtax your computer's resources


Create Output Filename & Directory

In [ ]:
# Set Output folder & filename prefix
filename = fire_name.replace(" ", "_")
os.makedirs(f"./Fires/{filename}", exist_ok=True)

print(filename)

Getting GOES JSON

In [ ]:
# # GeoServer WFS URL JSON configuration
# url = "https://sdge.sdsc.edu/geoserver/wfs"
# params = {
#     'service': 'WFS',
#     'version': '1.0.0',
#     'request': 'GetFeature',
#     'maxFeatures': '100000',
#     'exceptions': 'application/json',
#     'typeName': 'WIFIRE:ssec_wfabba_goes_no_low_code',
#     'CQL_FILTER': f"data_time BETWEEN '{dt_start_utc}' AND '{dt_end_utc}' AND BBOX(geom,{bbox})",
#     'outputFormat': 'application/json'
# }

# r = requests.get(url, params=params, timeout=60)
# if r.status_code != 200:
#     print(r.text)
#     sys.exit(1)
# elif r.status_code == 200:
#     j = r.json()    
#     with open(f"./Fires/{filename}/{filename}_goes.json", "w", encoding="utf-8") as file:
#         json.dump(j, file)
#     print("JSON saved successfully.")
# else:
#     print("Error:", r.status_code, r.text)

Getting GOES CSV

In [ ]:
# # GeoServer WFS URL CSV configuration
# url = "https://sdge.sdsc.edu/geoserver/wfs"
# params = {
#     'service': 'WFS',
#     'version': '1.0.0',
#     'request': 'GetFeature',
#     'maxFeatures': '100000',
#     'exceptions': 'application/json',
#     'typeName': 'WIFIRE:ssec_wfabba_goes_no_low_code',
#     'CQL_FILTER': f"data_time BETWEEN '{dt_start_utc}' AND '{dt_end_utc}' AND BBOX(geom,{bbox})",
#     'outputFormat': 'csv'
# }

# r = requests.get(url, params=params, timeout=60)
# if r.status_code != 200:
#     print(r.text)
#     sys.exit(1)
# elif r.status_code == 200:
#     with open(f"./Fires/{filename}/{filename}_goes.csv", "wb") as f:
#         f.write(r.content)
#     print("CSV saved successfully.")
# else:
#     print("Error:", r.status_code, r.text)

Getting GOES Shapefiles

In [ ]:
# # GeoServer WFS URL CSV configuration
# url = "https://sdge.sdsc.edu/geoserver/wfs"
# params = {
#     'service': 'WFS',
#     'version': '1.0.0',
#     'request': 'GetFeature',
#     'maxFeatures': '100000',
#     'exceptions': 'application/json',
#     'typeName': 'WIFIRE:ssec_wfabba_goes_no_low_code',
#     'CQL_FILTER': f"data_time BETWEEN '{dt_start_utc}' AND '{dt_end_utc}' AND BBOX(geom,{bbox})",
#     'outputFormat': 'shape-zip'
# }

# r = requests.get(url, params=params, timeout=60)
# if r.status_code != 200:
#     print(r.text)
#     sys.exit(1)
# elif r.status_code == 200:
#     with open(f"./Fires/{filename}/{filename}_goes.zip", "wb") as f:
#         for chunk in r.iter_content(chunk_size=1024*1024):
#             f.write(chunk)
#     print("Shapefile ZIP saved successfully.")
# else:
#     print("Error:", r.status_code, r.text)


